# Homework 4: Classifier Guidance for Flow Matching

**Deep Learning Course 2026**

**Author:** Nikita Kiselev

In this homework you will steer an **unconditional** Flow Matching model with an external classifier and measure what guidance does to sample quality.

| Part | Topic | Points |
|---|---|---|
| 1 | Unconditional Flow Matching | 1.5 |
| 2 | Quality metrics: FID, IS, Precision/Recall | 2.5 |
| 3 | Classifier on noisy data | 1.5 |
| 4 | Classifier guidance | 2.5 |
| 5 | Classifier guidance vs classifier-free guidance | 2 |

Fill in every `# YOUR CODE HERE` and every **Answer:** cell.

Do not change the given code: the checks rely on it.

Submit the notebook with all outputs through the Google Form.

**Deadline:** November 24, 2026.

In [ ]:
import hashlib
import torch
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from torchvision.utils import make_grid

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["font.size"] = 16

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

## 0. Setup

**Your variant**

Your dataset and five classes are set by your student ID.

Use your full name exactly as you enter it in the submission form: we recompute your variant from it.

In [ ]:
STUDENT_ID = "Name Surname"  # NOTE: put your full name here

In [ ]:
VARIANTS = [
    ("FashionMNIST", [0, 1, 2, 4, 9]),
    ("FashionMNIST", [0, 3, 4, 7, 8]),
    ("FashionMNIST", [0, 3, 6, 7, 8]),
    ("FashionMNIST", [1, 5, 7, 8, 9]),
    ("FashionMNIST", [2, 4, 5, 6, 7]),
    ("FashionMNIST", [2, 5, 6, 7, 9]),
    ("KMNIST", [0, 2, 3, 6, 8]),
    ("KMNIST", [0, 3, 5, 6, 9]),
    ("KMNIST", [0, 5, 7, 8, 9]),
    ("KMNIST", [1, 2, 3, 4, 9]),
    ("KMNIST", [3, 4, 5, 7, 9]),
    ("KMNIST", [4, 5, 6, 7, 9]),
]

variant = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % len(VARIANTS)
dataset_name, classes = VARIANTS[variant]
num_classes = len(classes)
print(f"Variant {variant}: {dataset_name}, classes {classes}")

**Data**

We keep only your five classes and relabel them as $0, \dots, 4$.

Pixels are scaled to $[-1, 1]$, as in the seminar.

In [ ]:
def load_split(train):
    ds = getattr(datasets, dataset_name)(root="./data", train=train, download=True)
    mask = torch.isin(ds.targets, torch.tensor(classes))
    x = ds.data[mask].unsqueeze(1).float() / 127.5 - 1  # [n, 1, 28, 28] in [-1, 1]
    y = torch.searchsorted(torch.tensor(classes), ds.targets[mask])  # labels 0..4
    return TensorDataset(x, y)

train_dataset = load_split(train=True)
test_dataset = load_split(train=False)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
print(f"train: {len(train_dataset)}, test: {len(test_dataset)}")

In [ ]:
def show(images, nrow=10, figsize=(12, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(make_grid(images.cpu(), nrow=nrow, normalize=True, value_range=(-1, 1)).permute(1, 2, 0))
    plt.axis("off")
    plt.show()

x, y = test_dataset.tensors
show(torch.cat([x[y == c][:10] for c in range(num_classes)]), figsize=(12, 6))

**Evaluator**

To measure quality we need a fixed feature extractor, like InceptionV3 for natural images.

We provide a small CNN trained on all ten classes of your dataset. Its weights are downloaded automatically.

The evaluator is for metrics only: do not train it and do not use it to guide sampling.

- `evaluator.features(x)` returns 128-dimensional features: use them for FID and Precision/Recall.
- `evaluator(x)` returns logits over the ten original classes: use them for IS and accuracy.

In [ ]:
class Evaluator(nn.Module):
    def __init__(self, num_classes=10, feature_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.GELU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.GELU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(64 * 7 * 7, feature_dim)
        self.head = nn.Linear(feature_dim, num_classes)

    def features(self, x):
        return F.gelu(self.fc(self.conv(x).flatten(1)))

    def forward(self, x):
        return self.head(self.features(x))

evaluator = Evaluator().to(device)
url = f"https://raw.githubusercontent.com/intsystems/Deep-Learning-Course/main/homeworks/homework-04/evaluator-{dataset_name.lower()}.pt"
evaluator.load_state_dict(torch.hub.load_state_dict_from_url(url, map_location=device))
evaluator.eval().requires_grad_(False)

The evaluator predicts original labels, so we map them back with `classes`. It should be accurate on your real test images:

In [ ]:
with torch.no_grad():
    pred = evaluator(x.to(device)).argmax(1).cpu()
print(f"evaluator accuracy on your test classes: {(pred == torch.tensor(classes)[y]).float().mean():.3f}")

## 1. Unconditional Flow Matching (1.5 points)

Train an **unconditional** Flow Matching model on your data: it ignores the labels.

We keep the seminar convention: $\mathbf{x}_0 \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ is noise, $\mathbf{x}_1$ is data, $\mathbf{x}_t = t \mathbf{x}_1 + (1 - t) \mathbf{x}_0$.

The architecture is up to you: an MLP as in the seminar works, a small convolutional U-Net works better. Keep the interface, the rest of the homework relies on it:

- `model(x, t)` takes images `x` of shape `[b, 1, 28, 28]` and times `t` of shape `[b, 1]`, and returns the velocity of the same shape as `x`;
- `sample(model, num_samples, num_steps)` returns images of shape `[num_samples, 1, 28, 28]` in $[-1, 1]$.

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.register_buffer("freqs", torch.arange(1, dim // 2 + 1) * torch.pi)

    def forward(self, t):
        emb = self.freqs * t
        return torch.cat([emb.cos(), emb.sin()], dim=-1)

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Training**

Store the loss of every iteration in `losses`.

In [ ]:
losses = []
# YOUR CODE HERE
raise NotImplementedError

In [ ]:
plt.plot(losses, linewidth=0.5)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.show()

**Sampling**

Use the Euler method from $t = 0$ to $t = 1$, as in the seminar.

In [ ]:
@torch.no_grad()
def sample(model, num_samples, num_steps=50):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
show(sample(model, 100))

## 2. Quality metrics (2.5 points)

All metrics below use the frozen evaluator. We compute features and logits once per set of images.

In [ ]:
@torch.no_grad()
def extract(images, batch_size=500):
    feats, logits = [], []
    for x in images.split(batch_size):
        f = evaluator.features(x.to(device))
        feats.append(f.cpu())
        logits.append(evaluator.head(f).cpu())
    return torch.cat(feats).double(), torch.cat(logits).double()

**Fréchet Inception Distance (FID)**

Fit a Gaussian to each set of features and compute the Fréchet distance between them:
$$
\mathrm{FID} = \left\| \boldsymbol{\mu}_r - \boldsymbol{\mu}_g \right\|_2^2 + \mathrm{Tr}\left( \boldsymbol{\Sigma}_r + \boldsymbol{\Sigma}_g - 2 \left( \boldsymbol{\Sigma}_r \boldsymbol{\Sigma}_g \right)^{1/2} \right)
$$

Lower is better. Mind the numerics: the matrix square root of a product of two covariance matrices may come out slightly complex.

In [ ]:
def fid(feats_a, feats_b):
    # YOUR CODE HERE
    raise NotImplementedError

**Inception Score (IS)**

A good sample has a confident prediction $p(y \mid \mathbf{x})$, and the samples together cover all classes, so the marginal $p(y)$ is broad:
$$
\mathrm{IS} = \exp\left( \mathbb{E}_{\mathbf{x}} \, \mathrm{KL}\left( p(y \mid \mathbf{x}) \,\|\, p(y) \right) \right)
$$

Split the samples into `splits` parts, compute IS on each part and return the mean and the standard deviation. Higher is better.

In [ ]:
def inception_score(logits, splits=10):
    # YOUR CODE HERE
    raise NotImplementedError

**Precision and Recall**

FID mixes two failures into one number: bad samples and missing modes. Precision and Recall [(Kynkäänniemi et al., 2019)](https://arxiv.org/abs/1904.06991) separate them.

Around every real feature vector, draw a ball whose radius is the distance to its $k$-th nearest real neighbour. The union of the balls approximates the real manifold. Then

- **Precision** is the fraction of generated samples that fall inside the real manifold;
- **Recall** is the fraction of real samples that fall inside the generated manifold, built the same way.

Use $k = 3$. A point is not its own neighbour.

In [ ]:
def precision_recall(feats_real, feats_fake, k=3):
    # YOUR CODE HERE
    raise NotImplementedError

**Sanity checks on real data**

Before we trust the metrics on generated samples, let's test them where we know the answer.

We take the test set as the "generated" set and the training set as the reference.

In [ ]:
x_train, y_train = train_dataset.tensors
x_test, y_test = test_dataset.tensors
feats_train, logits_train = extract(x_train[torch.randperm(len(x_train))[:5000]])
feats_test, logits_test = extract(x_test)

def report(name, feats, logits):
    is_mean, is_std = inception_score(logits)
    precision, recall = precision_recall(feats_train, feats)
    print(f"{name:>20}: FID {fid(feats_train, feats):7.2f}  IS {is_mean:.2f} ± {is_std:.2f}  "
          f"precision {precision:.3f}  recall {recall:.3f}")

report("real test", feats_test, logits_test)

(a) How does FID between two **real** sets depend on the number of samples $N$? Take two disjoint halves of the training set and plot FID against $N \in \{500, 1000, 2000, 5000\}$.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

(b) Drop two of your five classes from the test set.

(c) Replace a fraction $p \in \{0.1, 0.3, 0.5\}$ of the test images with pure noise $\mathcal{N}(\mathbf{0}, \mathbf{I})$, clipped to $[-1, 1]$.

Report all four metrics with `report` for each case.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

Finally, the metrics of your model from Part 1. We will compare guided models against this baseline in Part 5.

In [ ]:
samples = torch.cat([sample(model, 1000) for _ in range(5)])
report("unconditional FM", *extract(samples))

**Questions**

1. Why does FID between two real sets grow as $N$ decreases, although the distributions are the same?
2. Which metrics react to dropping classes, and which to junk? Why?
3. What is the largest possible IS in your setting, and does real data reach it? Why?

**Answer:**



## 3. Classifier on noisy data (1.5 points)

Classifier guidance needs $\nabla_{\mathbf{x}_t} \log p(y \mid \mathbf{x}_t)$ at **every** point of the sampling path, not only at clean images.

So we train a classifier $p_{\boldsymbol{\phi}}(y \mid \mathbf{x}_t, t)$ on noisy inputs $\mathbf{x}_t = t \mathbf{x}_1 + (1 - t) \mathbf{x}_0$ with $t \sim \mathcal{U}[0, 1]$.

Interface: `classifier(x, t)` returns logits over your five classes.

In [ ]:
class NoisyClassifier(nn.Module):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Accuracy along the path**

Plot the test accuracy against $t \in \{0, 0.1, \dots, 1\}$.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Question.** Explain the accuracy at both ends, $t = 0$ and $t = 1$.

**Answer:**



## 4. Classifier guidance (2.5 points)

In the lecture, classifier guidance adds the gradient of the classifier to the **score**:
$$
\mathbf{s}^{\gamma}(\mathbf{x}_t, t, y) = \mathbf{s}(\mathbf{x}_t, t) + \gamma \cdot \nabla_{\mathbf{x}_t} \log p(y \mid \mathbf{x}_t)
$$

Our model predicts a **velocity**. To guide it, we need a bridge between the two.

**Derivation.** For $\mathbf{x}_t = t \mathbf{x}_1 + (1 - t) \mathbf{x}_0$, show that

1. the optimal velocity $\mathbf{v}(\mathbf{x}_t, t) = \mathbb{E}[\mathbf{x}_1 - \mathbf{x}_0 \mid \mathbf{x}_t]$ gives $\mathbb{E}[\mathbf{x}_0 \mid \mathbf{x}_t] = \mathbf{x}_t - t \cdot \mathbf{v}(\mathbf{x}_t, t)$;
2. the score is $\mathbf{s}(\mathbf{x}_t, t) = -\mathbb{E}[\mathbf{x}_0 \mid \mathbf{x}_t] / (1 - t)$; *hint:* $\mathbf{x}_t \mid \mathbf{x}_1 \sim \mathcal{N}(t \mathbf{x}_1, (1 - t)^2 \mathbf{I})$;
3. hence the guided velocity is
   $$
   \mathbf{v}^{\gamma}(\mathbf{x}_t, t, y) = \mathbf{v}(\mathbf{x}_t, t) + \gamma \cdot \frac{1 - t}{t} \cdot \nabla_{\mathbf{x}_t} \log p(y \mid \mathbf{x}_t).
   $$

**Answer:**



**Guided velocity**

Implement $\mathbf{v}^{\gamma}$. The gradient comes from `torch.autograd.grad`: the classifier is frozen, but the input is not.

In [ ]:
def guided_velocity(model, classifier, x, t, y, gamma):
    # YOUR CODE HERE
    raise NotImplementedError

**Sampling with guidance**

The factor $(1 - t) / t$ explodes at $t = 0$, where every Euler path starts. Decide how to handle it and justify your choice with the plot from Part 3.

In [ ]:
@torch.no_grad()
def sample_cg(model, classifier, y, gamma, num_steps=50):
    # YOUR CODE HERE
    raise NotImplementedError

**Answer:**



Each row uses its own guidance scale $\gamma = 0, 1, 3, 10$; each pair of columns is one of your classes.

In [ ]:
y = torch.arange(num_classes, device=device).repeat_interleave(2)
show(torch.cat([sample_cg(model, classifier, y, gamma) for gamma in [0, 1, 3, 10]]), figsize=(12, 5))

## 5. Classifier guidance vs classifier-free guidance (2 points)

Now train a **conditional** model with label dropout, as in the seminar, and compare the two kinds of guidance with your metrics.

Use the same architecture as in Part 1 with a label input. Label `num_classes` means "no label".

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

In [ ]:
@torch.no_grad()
def sample_cfg(model, y, w, num_steps=50):
    # YOUR CODE HERE
    raise NotImplementedError

**Sweeps**

For each guidance scale, generate 1000 samples per class and compute FID, IS, Precision and Recall against the training features, and the evaluator accuracy: the fraction of samples whose evaluator prediction matches the requested class.

For classifier guidance, also record the mean confidence of **your** noisy classifier in the requested class at $t = 1$.

Store the results in `results[method][metric]` as lists over scales; the plot below expects exactly these keys.

In [ ]:
scales = [0, 1, 2, 4, 8, 16]
metrics = ["FID", "IS", "Precision", "Recall", "Accuracy"]
results = {"CG": {m: [] for m in metrics + ["Confidence"]}, "CFG": {m: [] for m in metrics}}
y = torch.arange(num_classes, device=device).repeat(1000)  # NOTE: interleave classes, so every IS split sees all of them

# YOUR CODE HERE
raise NotImplementedError

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))
for ax, m in zip(axes, metrics):
    for method in ["CG", "CFG"]:
        ax.plot(scales, results[method][m], marker="o", label=method)
    ax.set_title(m)
    ax.set_xlabel("Guidance scale")
axes[-1].plot(scales, results["CG"]["Confidence"], marker="o", linestyle="--", label="CG: own classifier")
axes[0].legend()
axes[-1].legend()
plt.tight_layout()
plt.show()

**Questions**

1. At a matched precision, which method has the better recall? Why?
2. For CFG, FID first improves and then degrades as $w$ grows. Why? Use Precision and Recall to explain.
3. Compare the confidence of your noisy classifier with the evaluator accuracy for classifier guidance at large $\gamma$. What does the gap tell you?

**Answer:**

